In [11]:
import pandas as pd
import numpy as np
import requests
import holidays
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [12]:
station = pd.read_csv("../../Data/sort_data/preprocessed_6station/ST-2425.csv")
station2025 = pd.read_csv("../../Data/sort_data/preprocessed_6station/ST-2425_2025.csv")

In [13]:
station.columns

Index(['기준_날짜', '시간대', '집계_기준', '시작_대여소_ID', '종료_대여소_ID', '전체_이용_분',
       '전체_이용_거리', '온도', '습도', '강수량', 'timestamp', 'station_id',
       'station_role', 'temp_lag_1hr', '위도', '경도', 'station_dong', 'year',
       'month_sin', 'month_cos', 'is_restingday', 'weekday_0', 'weekday_1',
       'weekday_2', 'weekday_3', 'weekday_4', 'weekday_5', 'weekday_6',
       'hour_sin', 'hour_cos', 'is_noon', 'is_rushhour', 'month',
       'residential_index', 'business_index', 'tourism_index', 'transit_index',
       'commute_in_index', 'commute_out_index', 'snow_flag'],
      dtype='object')

In [14]:
# 1. 출발 시간 도착 시간으로 변경
def update_end_time(df, station_id):
    df = df.copy()
    df["전체_이용_분"] = pd.to_timedelta(df["전체_이용_분"], errors="coerce")
    df["기준_날짜"] = pd.to_datetime(df["기준_날짜"])
    df["시간대"] = pd.to_numeric(df["시간대"], errors="coerce").astype("int64")

    mask = df["종료_대여소_ID"] == station_id
    hours = df.loc[mask, "전체_이용_분"].dt.total_seconds() / 3600.0

    raw = np.floor(df.loc[mask, "시간대"] - hours).astype("int64")
    day_shift = np.floor_divide(raw, 24).astype("int64")
    adj_hour = (raw - day_shift * 24).astype("int64")

    df.loc[mask, "기준_날짜"] += pd.to_timedelta(day_shift, unit="D")
    df.loc[mask, "시간대"] = adj_hour
    df.loc[mask, "집계_기준"] = "도착시간"

    return df.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)

# 2. 빈 시간대 없는 꽉 찬 시계열 만들기
def build_hourly_net(df):
    df = df.copy()
    df["기준_날짜"] = pd.to_datetime(df["기준_날짜"])

    # 전체 시간 범위 생성
    start = df["기준_날짜"].min()
    end = df["기준_날짜"].max()

    hours = pd.date_range(start, end + pd.Timedelta(hours=23), freq="h")

    base = pd.DataFrame({"_dt": hours})
    base["기준_날짜"] = base["_dt"].dt.floor("D")
    base["시간대"] = base["_dt"].dt.hour
    base = base.drop(columns="_dt")

    counts = (
        df.groupby(["기준_날짜", "시간대", "집계_기준"])
        .size()
        .unstack(fill_value=0)
    )

    base = base.merge(counts, on=["기준_날짜", "시간대"], how="left").fillna(0)

    # 순유입량 계산
    base["전체_건수"] = base.get("도착시간", 0) - base.get("출발시간", 0)

    return base.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)

# 3. 날씨 데이터 붙이기
def get_weather(start_date, end_date, lat=37.6, lon=126.93):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation",
        "timezone": "Asia/Seoul",
    }

    data = requests.get(url, params=params).json()["hourly"]

    weather = pd.DataFrame({
        "time": pd.to_datetime(data["time"]),
        "온도": data["temperature_2m"],
        "습도": data["relative_humidity_2m"],
        "강수량": data["precipitation"],
    })

    weather["기준_날짜"] = weather["time"].dt.floor("D")
    weather["시간대"] = weather["time"].dt.hour

    return weather.drop(columns="time")

def merge_weather(df, weather):
    return df.merge(weather, on=["기준_날짜", "시간대"], how="left")

# 주말 컬럼 추가
def add_holiday_features(df):
    df = df.copy()
    kr_holidays = holidays.KR()
    date = pd.to_datetime(df["기준_날짜"])

    df["is_holiday_or_weekend"] = (
        date.isin(kr_holidays) | (date.dt.dayofweek >= 5)
    ).astype(int)

    return df

# 요일 원핫 인코딩
def add_weekday_onehot(df):
    df = df.copy()
    
    df["dow"] = pd.to_datetime(df["기준_날짜"]).dt.dayofweek
    
    weekday_dummies = pd.get_dummies(df["dow"], prefix="weekday")
    
    df = pd.concat([df, weekday_dummies], axis=1)
    
    return df

# 피쳐 엔지니어링
def add_features(df):
    df = df.sort_values(["기준_날짜", "시간대"]).copy()
    date = pd.to_datetime(df["기준_날짜"])

    df["dow"] = date.dt.dayofweek
    df["month"] = date.dt.month

    df["hour_sin"] = np.sin(2*np.pi*df["시간대"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["시간대"]/24)

    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

    df["is_rush_hour"] = (
        df["시간대"].isin([7,8,18,19]) & (df["dow"] < 5)
    ).astype(int)

    df = add_weekday_onehot(df)
    df = add_holiday_features(df)

    return df

# 데이터셋 만들기
def make_dataset(df, station_id, weather):
    df = update_end_time(df, station_id)
    df = build_hourly_net(df)
    df = merge_weather(df, weather)

    return df

In [15]:
# 데이터 준비
weather = get_weather("2024-01-01", "2025-12-31")

station_ID = 'ST-2425'

train_df = make_dataset(station, station_ID, weather)
test_df = make_dataset(station2025, station_ID, weather)

train_df = add_features(train_df)
test_df = add_features(test_df)

# 피쳐 정하기
feature_cols = [
    "온도","습도","강수량",
    "weekday_0","weekday_1","weekday_2","weekday_3","weekday_4","weekday_5","weekday_6",
    "hour_sin","hour_cos","month_sin","month_cos",
    "is_holiday_or_weekend","is_rush_hour"
]

target_col = "전체_건수"

train_df = train_df.dropna(subset=feature_cols+[target_col])
test_df = test_df.dropna(subset=feature_cols)

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]


In [16]:
# CV
tscv = TimeSeriesSplit(n_splits=5)

cv_scores = []

for i, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=8,
        max_depth=4,
        min_child_samples=100,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=1.0,
        reg_lambda=1.0,
        random_state=42
    )

    model.fit(X_tr, y_tr)

    pred_tr = model.predict(X_tr)
    pred_val = model.predict(X_val)

    train_r2 = r2_score(y_tr, pred_tr)
    val_r2 = r2_score(y_val, pred_val)

    cv_scores.append(val_r2)

# 최종 모델
final_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=8,
    max_depth=4,
    min_child_samples=100,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=42
)

final_model.fit(X_train, y_train)

train_pred = final_model.predict(X_train)
test_pred = final_model.predict(X_test)

# 평가
def score(y, pred):
    return {
        "R2": r2_score(y, pred),
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred))
    }

print("\n===== FINAL SCORE =====")
print("TRAIN:", score(y_train, train_pred))
print("CV 평균 R2:", np.mean(cv_scores))
print("TEST :", score(y_test, test_pred))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000484 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 339
[LightGBM] [Info] Number of data points in the train set: 1464, number of used features: 16
[LightGBM] [Info] Start training from score -0.329918
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# -----------------------------
# 1. 데이터 준비
# -----------------------------
df = station.copy()

df["기준_날짜"] = pd.to_datetime(df["기준_날짜"])
df["month"] = df["기준_날짜"].dt.month

# -----------------------------
# 2. 월별 시간대 평균
# -----------------------------
monthly_hourly = (
    df.groupby(["month", "시간대"])["전체_건수"]
    .mean()
    .reset_index()
)

# -----------------------------
# 3. pivot (월 × 24시간)
# -----------------------------
pivot = monthly_hourly.pivot(index="month", columns="시간대", values="전체_건수")

# 시간대 정렬
pivot = pivot.reindex(columns=range(24))

# 결측 처리
pivot = pivot.fillna(0)

# -----------------------------
# 4. 시각화 (월별 패턴)
# -----------------------------
plt.figure(figsize=(12,6))

for m in pivot.index:
    plt.plot(pivot.columns, pivot.loc[m], label=f"{m}월")

plt.legend(ncol=3)
plt.xlabel("시간대")
plt.ylabel("평균 이용량")
plt.title("월별 시간대 패턴")
plt.show()

# -----------------------------
# 5. 스케일링 (중요)
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pivot)

# -----------------------------
# 6. 클러스터링
# -----------------------------
kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X_scaled)

pivot["cluster"] = labels

print("\n===== 월별 클러스터 =====")
print(pivot["cluster"].sort_index())

# -----------------------------
# 7. 클러스터별 패턴 시각화
# -----------------------------
for c in sorted(pivot["cluster"].unique()):
    cluster_data = pivot[pivot["cluster"] == c].drop(columns="cluster")

    plt.figure(figsize=(8,4))
    
    # 개별 월 (연하게)
    for idx in cluster_data.index:
        plt.plot(cluster_data.columns, cluster_data.loc[idx], alpha=0.3)

    # 평균 패턴 (굵게)
    plt.plot(cluster_data.columns, cluster_data.mean(), linewidth=3)

    plt.title(f"Cluster {c}")
    plt.xlabel("시간대")
    plt.ylabel("평균 이용량")
    plt.show()

KeyError: 'Column not found: 전체_건수'